<left>
<table style="margin-top:0px; margin-left:0px;">
<tr>
  <td><img src="https://raw.githubusercontent.com/worm-portal/WORM-Figures/master/style/worm.png" alt="WORM" title="WORM" width=50/></td>
  <td><h1 style=font-size:30px>How do limestone caves form?</h1><br />
</tr>
</table>
</left>

<img src="https://www.nps.gov/subjects/caves/images/CAVEFountain-Of-The-Fairies-Ronal-Kerbo.jpg" alt="A solution cave" title="A solution cave" width=100%/>

*Carlsbad Caverns cave formations. [National Park Service photo by Ronal C. Kerbo.](https://www.nps.gov/subjects/caves/solution-caves.htm)*

Meteoric water (rain and snow) contains some $\mathop{\rm{CO_2}}$ from the atmosphere. $\mathop{\rm{CO_2}}$ dissolving in pure water forms weak carbonic acid:

$$
\mathop{\rm{CO_{2(g)} + H_2O = HCO_3^- + H^+}}
$$

Since protons form, meteoric water is slightly acidic (pH 5 to 6). When meteoric water percolates into the ground, water-rock reactions can occur. For example, limestone can dissolve when in contact with meteoric water. The dominant minerals in limestone are calcium carbonates. This is frequently calcite, $\mathop{\rm{CaCO_3}}$.

A cave can form if enough limestone is dissolved.

Let's take a look at the kind of mineral dissolution reaction we might expect as a cave forms:

<!-- $$
\mathop{\rm{CaCO_{3(cr)}}}\limits_{calcite} + \mathop{\rm{H^+= Ca^{2+} + HCO_3^-}}
$$
 -->
$$
\mathop{\rm{CaCO_{3(cr)}}}\limits_{calcite} \mathop{\rm{= Ca^{2+} + CO_3^{2-}}}
$$

Carbonate from the dissolution of calcite is in equilibrium with bicarbonate and CO$_2$:

$$
\mathop{\rm{CO_3^{2-} + 2H^+ = HCO3^- + H^+ = H_2O + CO_{2(aq)}}}
$$

The pH of the fluid can be affected because $\mathop{\rm{H^+}}$ is involved in reaction. What do you expect to happen to the pH of the rainwater as calcite dissolves? Think about it for a moment, then let's model it!

---

### Step one: speciate our fluid

Our fluid is meteoric rainwater. Take a look at the input file `rainpH6_cave_input.csv`. It contains columns for our rainwater that describe variables like temperature, pressure, pH, and redox state (logfO2). There are also columns for concentrations of solutes dissolved in the rainwater: Cl$^-$, Ca$^{2+}$, and HCO$_3^-$.

Chloride has been given micromolal concentration to help balance out the positive charge from the protons of the pH 6 fluid.

Note that calcium and bicarbonate ions concentrations in the input file are negligably small (1E-18 molal). This is far below any detection limit for laboratory equipment. So why include them in the input file at all? We always need to include all the basis species necessary to perform a water-rock reaction calculation. Later, we will need to react this fluid with calcite, a mineral containing calcium and carbon. Therefore, we must include basis species for calcium (Ca$^{2+}$) and carbon (HCO$_3^-$) in our input file.

If we don't do this, then we will get an error when we try to run the water-rock calculation that says something like "calcite is not among the loaded minerals". If you ever see a message like this, remember to revisit your fluid input file and create columns for all the elements necessary for a reaction with your mineral(s) of interest.

All right, let's get this speciation started:

In [1]:
import aqequil
import pandas as pd

ae=aqequil.AqEquil(db="WORM", exclude_organics=True)

Loading Water-Organic-Rock-Microbe (WORM) thermodynamic databases...
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_latest.csv
wrm_data_latest.csv is now set as the active thermodynamic database.
Element database elements.csv is active.
Solid solution database solid_solutions.csv is active.
LogK database wrm_data_logK.csv is active.
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_logK.csv
LogK_S database wrm_data_logK_S.csv is active.
Excluding ['organic_aq', 'organic_cr'] from column 'category_1'in wrm_data_logK_S.csv
Loading thermodynamic database into pyCHNOSZ...


In [2]:
speciation = ae.speciate(
                         input_filename="rainpH6_cave_input.csv",
                         charge_balance_on="H+",
                         )

The input file column 'logfO2' will be used to set sample redox state. If a another column is desired, set it manually using the redox_flag parameter.
Getting wrm_data_latest.csv ready. This will take a moment...
Using wrm_data_latest.csv to speciate rain_pH6
Finished!


The speciation is complete. We allowed the calculation to balance fluid charge on pH, so let's make sure our rainwater is still approximately pH 6:

In [3]:
speciation.lookup('pH')

Sample,pH
,pH
Sample,
rain_pH6,5.9962


If the pH was no longer around six, then we could go into our input file and change the concentration of chloride. Or we could simply balance charge on chloride instead of pH by setting `charge_balance_on="Cl-"` in the speciation function. Either way is valid.

### Step two: react fluid with rock

Now that we have speciated our rainwater, we can react it with calcite mineral.

Let's create a reactant for 1 mole of calcite and then prepare the reaction:

In [4]:
Cal = aqequil.Reactant(reactant_name="calcite",
                       amount_remaining=1) # moles of mineral to react per kg H2O

r = aqequil.Prepare_Reaction(reactants=[Cal])

The next step reacts the rainwater with calcite:

In [5]:
speciation = aqequil.react(speciation, r)

Using wrm_data_latest.csv to react rain_pH6


We only have one fluid sample in this demo: `rain_pH6`. Let's select it so we can examine it closer with diagrams:

In [6]:
m = speciation.mt("rain_pH6")

First, let's plot the volume of calcite remaining as 1kg of rainwater reacts with 1 mole of mineral: 

In [7]:
m.plot_product_minerals(show_reactant_minerals=True, log_y=False, y_type='volume')

There is a small decrease in the volume of 1 mole of our calcite reactant before the fluid becomes saturated with respect to calcite. **A tiny cave has formed according to our model!**

In this demo, Xi represents the number of moles of calcite that has reacted with the fluid at a given point in the calculation. The calculation ends once the fluid is saturated with respect to calcite after approximately 1E-4 moles of calcite dissolves into the fluid (log Xi = -4).

As calcite dissolves, the concentration of calcium and carbon in the rainwater increases:

In [8]:
m.plot_elements(log=True, plot_elements=["Ca", "C"])

(The dissolution of calcite causes a 1:1 increase in Ca and C concentration, leading to overlapping lines.)

We can also visualize what happens to the composition of the fluid in multivariate space by plotting a reaction path:

In [9]:
e = m.plot_reaction_paths()
e[0].show()

There are is a very small number of variables in our simplified system, so only one possible plot has been created. Our rainwater started at extremely low concentrations of Ca$^{2+}$ and HCO$_3^-$, as shown by the start of the reaction path in the lower left corner. As the reaction with calcite proceeds, the concentrations of both Ca$^{2+}$, and HCO$_3^-$ increase on a trajectory toward the dashed calcite saturation line.

There is a crook in the reaction path before the fluid reaches calcite saturation. Why do you think this happens? Note that the axes of the reaction path diagram are expressed as activity ratios with protons in denominator. Maybe pH has something to do with this!

Finally, let's plot what happens to the pH of our rainwater as it gradually reacts with calcite:

In [10]:
m.plot_pH()

As calcite dissolves, the pH of the fluid increases dramatically from six to around ten! That represents four orders of magnitude decrease in proton activity.

Carbonate released from the dissolution of calcite associates with protons to form bicarbonate where it can. This effectively removes protons from the fluid and drives the pH up.

In real life, saturated meteoric fluid in contact with calcite might percolate deeper and be replenished by fresh rainwater, eventually leading to the formation of enormous limestone caverns.

End of demo.